# Additional Feature Extraction (local)

This notebook runs **locally** (no Google Colab / `drive.mount`). It is designed so you can:

1. **Reuse already-processed features** — load `all_features_combined.pickle`, `dnn_feature.pickle`,
   the per-clip behavior CSVs (`behavior_results.zip`), and labels (`label_df_3.csv`) without recomputing anything.
2. **Download raw data on demand** — only fetch the original GEHM video / OpenPose keypoints / TextGrid
   for a meeting when a *new* feature actually needs them. Downloads are cached under `raw_data/` and skipped if present.
3. **Compute a new feature per segment** and merge it into an **extended** copy of the feature table
   (originals are never overwritten).

Run the cells top-to-bottom. Edit the `compute_extra_feature(...)` function in the
"Compute additional feature" section to implement your own feature.

In [1]:
# --- Optional: install deps (uncomment on a fresh environment) ---
# %pip install pandas numpy scikit-learn opencv-python pillow requests tqdm tgt
# Speaker (audio + text) features:
# %pip install praat-parselmouth vaderSentiment soundfile scipy
# CLIP (only needed if a new feature uses the DNN encoder):
# %pip install git+https://github.com/openai/CLIP.git torch

In [2]:
import os
import io
import re
import json
import zipfile
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

# --- Paths (relative to this notebook / repo root) ---
REPO_ROOT = Path.cwd()                       # run the notebook from the repo root
RAW_DIR = REPO_ROOT / "raw_data"             # downloaded raw video/keypoints/textgrid cache
BEHAVIOR_DIR = REPO_ROOT / "behavior_results"  # per-clip CSVs (extracted from behavior_results.zip)
RAW_DIR.mkdir(exist_ok=True)
print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = /Users/ducanh/Desktop/CCS2_code


## 1. Extract & load preprocessed artifacts (no reprocessing)

In [3]:
def ensure_extracted(zip_name, target_dir):
    """Unzip <zip_name> into the repo root once; skip if target_dir already exists."""
    target = REPO_ROOT / target_dir
    if target.exists():
        print(f"[skip] {target_dir} already extracted")
        return target
    zpath = REPO_ROOT / zip_name
    if not zpath.exists():
        print(f"[warn] {zip_name} not found; cannot extract {target_dir}")
        return target
    with zipfile.ZipFile(zpath) as z:
        z.extractall(REPO_ROOT)
    print(f"[ok] extracted {zip_name} -> {target_dir}")
    return target

ensure_extracted("behavior_results.zip", "behavior_results")
ensure_extracted("reference_behavior_results.zip", "reference_behavior_results")

[skip] behavior_results already extracted
[skip] reference_behavior_results already extracted


PosixPath('/Users/ducanh/Desktop/CCS2_code/reference_behavior_results')

In [4]:
# Vectorized master feature table (one row per segment_id)
with open(REPO_ROOT / "all_features_combined.pickle", "rb") as f:
    all_features_combined_df = pickle.load(f)

# Per-segment CLIP embeddings: dict { "<meeting>-<person>/clip_s_e.mp4": np.ndarray(512,) }
with open(REPO_ROOT / "dnn_feature.pickle", "rb") as f:
    dnn_feature = pickle.load(f)

# Human engagement labels
label_df_3 = pd.read_csv(REPO_ROOT / "label_df_3.csv")

# Clip boundaries + target<->reference mapping
target_segmentation = pd.read_csv(REPO_ROOT / "target_segmentation.csv")

print("all_features_combined_df:", all_features_combined_df.shape, list(all_features_combined_df.columns))
print("dnn_feature entries:", len(dnn_feature))
print("label_df_3:", label_df_3.shape)
print("target_segmentation:", target_segmentation.shape)
all_features_combined_df.head(3)

all_features_combined_df: (549, 10) ['segment_id', 'AU_feature', 'dnn_feature', 'manual_feature', 'AU_feature_ref1', 'dnn_feature_ref1', 'manual_feature_ref1', 'AU_feature_ref2', 'dnn_feature_ref2', 'manual_feature_ref2']
dnn_feature entries: 638
label_df_3: (102, 10)
target_segmentation: (631, 8)


,segment_id,AU_feature,dnn_feature,manual_feature,AU_feature_ref1,dnn_feature_ref1,manual_feature_ref1,AU_feature_ref2,dnn_feature_ref2,manual_feature_ref2
0,20210323-SP07F_clip_378_383,"[0.1441269841269841, 0.3001846627428381, 2.0, ...","[0.034613874, -0.22478016, 0.07269942, -0.3664...","[126.0, 12.609585210300457, 28.444309351049924...","[0.2984126984126984, 0.581175928886822, 3.0, 3...","[0.052595716, 0.06267054, -0.123, -0.41390204,...","[126.0, -23.982031966138386, 25.88700200677383...","[0.1348412698412698, 0.2224328541451883, 6.0, ...","[-0.299179, -0.36308083, -0.21096529, 0.192069...","[126.0, -3.078479217421869, 24.666117899730004..."
1,20210323-SP07F_clip_1113_1118,"[0.1441269841269841, 0.3001846627428381, 2.0, ...","[0.061545894, -0.25375453, -0.04545928, -0.193...","[125.0, 8.967601040954058, 26.538118969393903,...","[0.053015873015873, 0.0807590969862327, 0.0, -...","[0.07015944, 0.020164885, -0.05986896, -0.2490...","[126.0, -42.03995220760016, 27.480209132219365...","[0.2984126984126984, 0.581175928886822, 3.0, 3...","[0.027060911, -0.08274244, 0.0400747, 0.069514...","[126.0, -1.0203919666780152, 26.30721485617076..."
2,20210323-SP07F_clip_1074_1079,"[0.1441269841269841, 0.3001846627428381, 2.0, ...","[-0.009148686, -0.12560014, -0.08919125, -0.21...","[121.0, 16.190503685472198, 26.270694432031604...","[0.1348412698412698, 0.2224328541451883, 6.0, ...","[-0.299179, -0.36308083, -0.21096529, 0.192069...","[126.0, 22.120582578287895, 22.3158813791212, ...","[0.2984126984126984, 0.581175928886822, 3.0, 3...","[0.037613317, -0.005479859, 0.007825111, -0.00...","[126.0, -3.270940913024445, 27.572546816724127..."


## 2. Segment-ID helpers

A `segment_id` looks like `20230310-SP03M_clip_75_80`. These helpers map it to the meeting/person, the clip times, and the various on-disk / in-memory keys.

In [5]:
SEG_RE = re.compile(r"^(?P<meeting>\d+)-(?P<person>[A-Z0-9]+)_clip_(?P<start>\d+)_(?P<end>\d+)$")

def parse_segment_id(segment_id):
    m = SEG_RE.match(segment_id)
    if not m:
        raise ValueError(f"Unrecognized segment_id: {segment_id}")
    d = m.groupdict()
    return {
        "meeting": d["meeting"],
        "person": d["person"],
        "start": int(d["start"]),
        "end": int(d["end"]),
        "person_folder": f"{d['meeting']}-{d['person']}",      # e.g. 20230310-SP03M
        "clip_name": f"clip_{d['start']}_{d['end']}.mp4",
    }

def behavior_csv_path(segment_id):
    """Path to the already-extracted per-clip behavior CSV (raw manual + AU features)."""
    p = parse_segment_id(segment_id)
    return BEHAVIOR_DIR / p["person_folder"] / f"clip_{p['start']}_{p['end']}.csv"

def dnn_key(segment_id):
    """Key into the dnn_feature dict."""
    p = parse_segment_id(segment_id)
    return f"{p['person_folder']}/{p['clip_name']}"

def load_behavior_row(segment_id):
    """Load the per-clip raw feature CSV (single row) if it was already processed."""
    path = behavior_csv_path(segment_id)
    if not path.exists():
        return None
    return pd.read_csv(path).iloc[0]

# sanity check
sid = all_features_combined_df["segment_id"].iloc[0]
print("example:", sid, "->", parse_segment_id(sid))
print("behavior csv exists:", behavior_csv_path(sid).exists())
print("dnn key present:", dnn_key(sid) in dnn_feature)

example: 20210323-SP07F_clip_378_383 -> {'meeting': '20210323', 'person': 'SP07F', 'start': 378, 'end': 383, 'person_folder': '20210323-SP07F', 'clip_name': 'clip_378_383.mp4'}
behavior csv exists: True
dnn key present: True


## 3. Download raw data on demand (cached)

The original GEHM corpus is hosted on archive.org. Only call `download_raw_for_meeting(...)`
when a new feature needs the raw video / keypoints / TextGrid — already-downloaded files are skipped.

Layout written under `raw_data/`:
```
raw_data/<meeting>/<meeting>-<person>.mp4         separated speaker video
raw_data/<meeting>/openpose/                       extracted OpenPose keypoints (from <meeting>.zip)
raw_data/<meeting>/<meeting>_Praat_long.TextGrid   transcript tiers (per-participant)
raw_data/<meeting>/<meeting>-<person>.flac         separated speaker audio
```

In [6]:
import requests
from tqdm.auto import tqdm

ARCHIVE_BASE = "https://archive.org/download/GEHM_meeting_corpus/corpus"

def _download(url, dest):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {dest.name} already downloaded")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)
                bar.update(len(chunk))
    return dest

def download_raw_for_meeting(meeting, persons=None, what=("video", "keypoints", "textgrid")):
    """Fetch raw assets for a meeting. `persons` limits which speaker videos to pull.
    `what` selects asset types. Everything is cached under raw_data/<meeting>/."""
    out = RAW_DIR / meeting
    out.mkdir(parents=True, exist_ok=True)

    if "video" in what and persons:
        for person in persons:
            name = f"{meeting}-{person}.mp4"
            _download(f"{ARCHIVE_BASE}/{meeting}/SeparatedSpeakersVideo/{name}", out / name)

    if "audio" in what and persons:
        for person in persons:
            name = f"{meeting}-{person}.flac"
            try:
                _download(f"{ARCHIVE_BASE}/{meeting}/SeparatedSpeakersAudio/{name}", out / name)
            except:
                _download(f"{ARCHIVE_BASE}/{meeting}/SeparateSpeakersAudio/{name}", out / name)

    if "keypoints" in what:
        kp_zip = out / f"{meeting}.zip"
        _download(f"{ARCHIVE_BASE}/{meeting}/OpenPoseKeypoints/{meeting}.zip", kp_zip)
        kp_dir = out / "openpose"
        if not kp_dir.exists():
            with zipfile.ZipFile(kp_zip) as z:
                z.extractall(kp_dir)
            print(f"[ok] extracted keypoints -> {kp_dir}")

    if "textgrid" in what:
        tg = f"{meeting}_Praat_long.TextGrid"
        _download(f"{ARCHIVE_BASE}/{meeting}/{tg}", out / tg)

    return out

def raw_video_path(segment_id):
    """Local path to the speaker video for a segment (download first if missing)."""
    p = parse_segment_id(segment_id)
    return RAW_DIR / p["meeting"] / f"{p['person_folder']}.mp4"

def raw_audio_path(meeting, person):
    """Local path to a participant's separated-speaker audio (download first if missing)."""
    return RAW_DIR / meeting / f"{meeting}-{person}.flac"

# Example (uncomment to actually download):
download_raw_for_meeting("20210323", persons=["SP07F"], what=("video",))

[skip] 20210323-SP07F.mp4 already downloaded


/Users/ducanh/miniconda3/envs/thesis2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/ducanh/Desktop/CCS2_code/raw_data/20210323')

## 4. Compute an additional feature

Implement your feature in `compute_extra_feature(segment_id)`. It can use:
- `load_behavior_row(segment_id)` — already-extracted raw manual/AU values (no raw download needed),
- `dnn_feature[dnn_key(segment_id)]` — the cached CLIP embedding,
- `raw_video_path(segment_id)` + `download_raw_for_meeting(...)` — only if you need the raw frames.

The example below derives a simple scalar from already-processed values, so it needs **no** download.
Replace the body with your own logic (and call the downloader if you need raw media).

In [7]:
import cv2  # only needed if your feature reads raw video frames

def compute_extra_feature(segment_id):
    """Return a feature value (scalar / list / np.ndarray) for one segment.
    Return None to leave the segment unset."""
    row = load_behavior_row(segment_id)
    if row is None:
        return None

    # --- EXAMPLE: ratio of total head movement to clip duration -----------------
    # (uses only already-processed values; no raw video required)
    total_move = float(row.get("total_yaw_movement", 0)) \
               + float(row.get("total_pitch_movement", 0)) \
               + float(row.get("total_roll_movement", 0))
    duration = max(float(row.get("duration", 5)), 1e-6)
    return total_move / duration

    # --- If you DO need raw frames, the pattern is: -----------------------------
    # p = parse_segment_id(segment_id)
    # download_raw_for_meeting(p["meeting"], persons=[p["person"]], what=("video",))
    # cap = cv2.VideoCapture(str(raw_video_path(segment_id)))
    # ... read frames between p["start"] and p["end"] ...

# Smoke-test on the first few segments
for sid in all_features_combined_df["segment_id"].head(3):
    print(sid, "->", compute_extra_feature(sid))

20210323-SP07F_clip_378_383 -> 1051.7661043069752
20210323-SP07F_clip_1113_1118 -> 54.524506279919066
20210323-SP07F_clip_1074_1079 -> 627.9172419513583


## 5. Merge into an *extended* table and save

The new column is added to a copy and written to a new pickle/csv — the original `all_features_combined.pickle` is left untouched.

In [8]:
# FEATURE_NAME = "head_movement_rate"   # name your new column

# extended_df = all_features_combined_df.copy()
# extended_df[FEATURE_NAME] = extended_df["segment_id"].map(compute_extra_feature)

# n_missing = extended_df[FEATURE_NAME].isna().sum()
# print(f"computed '{FEATURE_NAME}' for {len(extended_df) - n_missing}/{len(extended_df)} segments "
#       f"({n_missing} missing)")

# out_pickle = REPO_ROOT / "all_features_combined_extended.pickle"
# out_csv = REPO_ROOT / "all_features_combined_extended.csv"
# with open(out_pickle, "wb") as f:
#     pickle.dump(extended_df, f)
# extended_df.to_csv(out_csv, index=False)
# print("saved ->", out_pickle.name, "and", out_csv.name)
# extended_df[["segment_id", FEATURE_NAME]].head()

## 6. Speaker-side features (who is talking *to* the listener)

The target person in every segment is a **listener** (chosen because they are *not* speaking). To predict their
engagement, we also characterize the **speaker** addressing them during the same window. Two independent feature
sets are extracted (one function each):

- **Acoustic** — prosody from the speaker's raw separated audio (`.flac`) via Praat/parselmouth.
- **Text** — lightweight, interpretable features from the speaker's transcript words (English).

First we identify the speaker from the meeting TextGrid: each participant has a word-level tier, so the speaker
is the *other* participant whose tier has the most speech overlapping the segment window.

In [9]:
import tgt

_textgrid_cache = {}

def load_textgrid(meeting):
    """Parse <meeting>_Praat_long.TextGrid (download on demand, cache per meeting)."""
    if meeting in _textgrid_cache:
        return _textgrid_cache[meeting]
    path = RAW_DIR / meeting / f"{meeting}_Praat_long.TextGrid"
    if not path.exists():
        download_raw_for_meeting(meeting, what=("textgrid",))
    tg = tgt.io.read_textgrid(str(path))
    _textgrid_cache[meeting] = tg
    return tg

def _overlap(a0, a1, b0, b1):
    return max(0.0, min(a1, b1) - max(a0, b0))

def _tier_speech_intervals(tier, start, end):
    """Non-empty, valid intervals of `tier` overlapping [start, end]
    (the TextGrids contain spurious [-1,-1] marker intervals, filtered here)."""
    return [iv for iv in getattr(tier, "intervals", [])
            if iv.text.strip() and iv.end_time > iv.start_time >= 0
            and _overlap(start, end, iv.start_time, iv.end_time) > 0]

def find_speaker(segment_id):
    """Primary speaker during a listener's segment: the participant (other than the
    target) whose tier has the most speech overlapping the window.
    Returns (speaker_person_id, overlap_seconds); (None, 0.0) if nobody speaks."""
    p = parse_segment_id(segment_id)
    tg = load_textgrid(p["meeting"])
    best = (None, 0.0)
    for tier in tg.tiers:
        if tier.name == p["person"]:
            continue
        ov = sum(_overlap(p["start"], p["end"], iv.start_time, iv.end_time)
                 for iv in _tier_speech_intervals(tier, p["start"], p["end"]))
        if ov > best[1]:
            best = (tier.name, ov)
    return best

# quick check (downloads the meeting TextGrid if absent)
_demo = label_df_3["segment_id"].iloc[0]
print(_demo, "-> speaker", find_speaker(_demo))

20210323-SP01F_clip_732_737 -> speaker ('SP06M', 2.6579999999996744)


### 6a. Acoustic features — behaviorally-grounded prosodic *events* (parselmouth)

Aggregate prosody stats are blunt at 5 s. Instead we count interactionally meaningful events on the speaker's
F0 contour, converting voiced F0 to **semitones relative to the window's median pitch** (so rises/accents are
speaker-independent). Key cues: **pitch-rise events** and a **terminal rise** (`final_rise_st > 0`) signal
question/engagement-eliciting intonation — important here because the transcripts have *no* punctuation.

In [10]:
import soundfile as sf
import parselmouth
from parselmouth.praat import call
from scipy.signal import find_peaks

_AC_KEYS = ["f0_mean", "f0_std_st", "pitch_rise_count", "pitch_rise_rate", "pitch_fall_count",
            "final_rise_st", "f0_slope_st", "f0_curv", "accent_count", "intensity_mean",
            "intensity_slope", "intensity_peak_count", "pause_count", "pause_ratio", "voiced_rate",
            "articulation_rate", "jitter", "shimmer", "hnr"]
RISE_ST = 2.0  # semitone threshold for a rise/fall event

def _f0_events(times, st):
    """Count rise/fall excursions (>= RISE_ST semitones) and prominent F0 peaks over voiced runs."""
    rises = falls = accents = 0
    if len(times) >= 2:
        runs = np.split(np.arange(len(times)), np.where(np.diff(times) > 0.08)[0] + 1)
    else:
        runs = [np.arange(len(times))] if len(times) else []
    for r in runs:
        if len(r) < 3:
            continue
        seg = st[r]; cum = 0.0
        for x in np.diff(seg):
            cum = (max(cum, 0) + x) if x > 0 else (min(cum, 0) + x)
            if cum >= RISE_ST: rises += 1; cum = 0.0
            elif cum <= -RISE_ST: falls += 1; cum = 0.0
        accents += len(find_peaks(seg, prominence=RISE_ST)[0])
    return rises, falls, accents

def compute_speaker_acoustic_features(segment_id, f0min=75, f0max=400):
    """Event-based prosody + voice-quality of the speaker's windowed speech (semitones rel. window median)."""
    p = parse_segment_id(segment_id)
    speaker, ov = find_speaker(segment_id)
    if speaker is None:
        return {**{k: np.nan for k in _AC_KEYS}, "has_speaker": False}

    download_raw_for_meeting(p["meeting"], persons=[speaker], what=("audio",))
    flac = raw_audio_path(p["meeting"], speaker)
    sr = sf.info(str(flac)).samplerate
    y, _ = sf.read(str(flac), start=int(p["start"]*sr), stop=int(p["end"]*sr), dtype="float64")
    if y.ndim > 1:
        y = y.mean(axis=1)
    dur = max(len(y) / sr, 1e-6)

    snd = parselmouth.Sound(y, sampling_frequency=sr)
    pitch = snd.to_pitch(pitch_floor=f0min, pitch_ceiling=f0max)
    f0 = pitch.selected_array["frequency"]; pt = pitch.xs()
    vmask = f0 > 0; vf0 = f0[vmask]; vt = pt[vmask]
    inten = snd.to_intensity(); ivals = inten.values[0]; itimes = inten.xs()
    fin = np.isfinite(ivals); iv = ivals[fin]; it = itimes[fin]

    out = {**{k: 0.0 for k in _AC_KEYS}, "has_speaker": True}
    for k in ["f0_mean", "f0_std_st", "pause_ratio", "jitter", "shimmer", "hnr"]:
        out[k] = np.nan
    out["intensity_mean"] = float(np.mean(iv)) if iv.size else np.nan

    if vf0.size:
        st = 12.0 * np.log2(vf0 / np.median(vf0))          # semitones rel. window median
        out["f0_mean"] = float(np.mean(vf0)); out["f0_std_st"] = float(np.std(st))
        rises, falls, accents = _f0_events(vt, st)
        out["pitch_rise_count"] = int(rises); out["pitch_fall_count"] = int(falls)
        out["pitch_rise_rate"] = float(rises / max(ov, dur)); out["accent_count"] = int(accents)
        if vt.size >= 3:
            out["f0_slope_st"] = float(np.polyfit(vt - vt[0], st, 1)[0])
            tail = vt >= (vt[-1] - 0.5)
            if tail.sum() >= 3:
                out["final_rise_st"] = float(np.polyfit(vt[tail] - vt[tail][0], st[tail], 1)[0])
        if vt.size >= 4:
            out["f0_curv"] = float(np.polyfit(vt - vt[0], st, 2)[0])   # contour curvature (rise-fall shape)
        gaps = np.diff(vt)
        out["pause_count"] = int(np.sum(gaps >= 0.2))
        out["voiced_rate"] = float((np.sum(gaps > 0.08) + 1) / max(ov, dur))
        out["pause_ratio"] = float(1.0 - vf0.size / max(f0.size, 1))

    if iv.size:
        if np.std(iv) > 0:
            # intensity peaks ~ syllable nuclei -> articulation rate (nuclei per voiced second)
            nuclei, _ = find_peaks(iv, prominence=float(np.std(iv)), distance=max(1, int(0.12 / (it[1]-it[0]) if it.size > 1 else 1)))
            out["intensity_peak_count"] = int(len(nuclei))
            voiced_dur = float(vt[-1] - vt[0]) if vt.size >= 2 else dur
            out["articulation_rate"] = float(len(nuclei) / max(voiced_dur, 1e-6))
        if it.size >= 3:
            out["intensity_slope"] = float(np.polyfit(it - it[0], iv, 1)[0])

    # voice quality via Praat (robust to failures on short/unvoiced audio)
    try:
        pp = call(snd, "To PointProcess (periodic, cc)", f0min, f0max)
        out["jitter"] = float(call(pp, "Get jitter (local)", 0, 0, 1e-4, 0.02, 1.3))
        out["shimmer"] = float(call([snd, pp], "Get shimmer (local)", 0, 0, 1e-4, 0.02, 1.3, 1.6))
    except Exception:
        pass
    try:
        out["hnr"] = float(call(snd.to_harmonicity_cc(), "Get mean", 0, 0))
    except Exception:
        pass
    return out

### 6b. Text features — interactional lexicon (English)

Counts grounded in the corpus vocabulary: **question cues** (wh-words / utterance-initial auxiliaries — there is
no punctuation to rely on), **backchannel / feedback** tokens (yeah, right, okay, sure, exactly, mhm...),
**fillers** (uh, um, like, well, "you know"...), negation, plus directed "you" / inclusive "we" rates and VADER
sentiment.

In [11]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_vader = SentimentIntensityAnalyzer()
_BACKCHANNEL = {"yeah","yes","yep","yup","mhm","mm","mmm","hmm","uhhuh","okay","ok","right",
                "sure","exactly","true","indeed","absolutely","totally","good","nice","cool"}
_FILLER = {"uh","um","er","erm","like","well"}
_FILLER_BIGRAMS = ["you know", "i mean", "sort of", "kind of"]
_NEGATION = {"no","not","never","dont","cant","nope","cannot","wont","didnt","doesnt"}
_WH  = {"what","why","how","when","where","who","which","whom","whose"}
_AUX = {"do","does","did","is","are","am","was","were","can","could","would","will","shall",
        "should","have","has","had","may","might"}
_SECOND = {"you","your","yours","youre","you're"}
_FIRSTPL = {"we","us","our","ours"}
_TX_KEYS = ["word_count","words_per_sec","question_cue_count","starts_question","backchannel_count",
            "backchannel_rate","filler_rate","negation_rate","second_person_rate",
            "first_plural_rate","sent_compound"]

def speaker_words(segment_id):
    """Time-ordered lowercase words the speaker utters within the segment window."""
    p = parse_segment_id(segment_id)
    speaker, _ = find_speaker(segment_id)
    if speaker is None:
        return None, None
    tier = next(t for t in load_textgrid(p["meeting"]).tiers if t.name == speaker)
    ivs = sorted(_tier_speech_intervals(tier, p["start"], p["end"]), key=lambda x: x.start_time)
    return speaker, [iv.text.strip().lower() for iv in ivs]

def compute_speaker_text_features(segment_id):
    """Interactional English lexicon features of the speaker's utterance."""
    speaker, words = speaker_words(segment_id)
    _, ov = find_speaker(segment_id)
    if not words:
        return {**{k: 0.0 for k in _TX_KEYS}, "has_speaker": False}
    toks = [w for word in words for w in re.findall(r"[a-z']+", word)]
    n = len(toks) or 1
    text = " ".join(toks)
    bigram_fill = sum(text.count(b) for b in _FILLER_BIGRAMS)
    return {
        "has_speaker": True,
        "word_count": len(toks),
        "words_per_sec": len(toks) / max(ov, 1e-6),
        "question_cue_count": sum(t in _WH for t in toks),
        "starts_question": float(bool(toks) and (toks[0] in _WH or toks[0] in _AUX)),
        "backchannel_count": sum(t in _BACKCHANNEL for t in toks),
        "backchannel_rate": sum(t in _BACKCHANNEL for t in toks) / n,
        "filler_rate": (sum(t in _FILLER for t in toks) + bigram_fill) / n,
        "negation_rate": sum(t in _NEGATION for t in toks) / n,
        "second_person_rate": sum(t in _SECOND for t in toks) / n,
        "first_plural_rate": sum(t in _FIRSTPL for t in toks) / n,
        "sent_compound": _vader.polarity_scores(text)["compound"],
    }

### 6c. Test on ONE segment first

Validate the whole chain (speaker id -> audio download -> acoustic + text) on a single labeled segment
before running the batch over all segments.

In [12]:
test_sid = label_df_3["segment_id"].iloc[0]
spk, ov = find_speaker(test_sid)
_, _words = speaker_words(test_sid)
print("segment  :", test_sid, " (listener =", parse_segment_id(test_sid)["person"], ")")
print("speaker  :", spk, f"({ov:.2f}s overlap)")
print("utterance:", " ".join(_words) if _words else None)
print()
print("text features    :", compute_speaker_text_features(test_sid))
print()
print("acoustic features:", compute_speaker_acoustic_features(test_sid))

segment  : 20210323-SP01F_clip_732_737  (listener = SP01F )
speaker  : SP06M (2.66s overlap)
utterance: these research questions when youre talking about them in these meetings so

text features    : {'has_speaker': True, 'word_count': 12, 'words_per_sec': 4.514672686230801, 'question_cue_count': 1, 'starts_question': 0.0, 'backchannel_count': 0, 'backchannel_rate': 0.0, 'filler_rate': 0.0, 'negation_rate': 0.0, 'second_person_rate': 0.08333333333333333, 'first_plural_rate': 0.0, 'sent_compound': 0.0}

[skip] 20210323-SP06M.flac already downloaded
acoustic features: {'f0_mean': 111.15654631544437, 'f0_std_st': 5.125526782871861, 'pitch_rise_count': 1, 'pitch_rise_rate': 0.2, 'pitch_fall_count': 4, 'final_rise_st': 4.787621394048952, 'f0_slope_st': 1.1295957642406003, 'f0_curv': -0.159900118551172, 'accent_count': 0, 'intensity_mean': 55.491168982936074, 'intensity_slope': -2.065228781506792, 'intensity_peak_count': 16, 'pause_count': 3, 'pause_ratio': 0.6700201207243461, 'voiced_rate':

### 6d. Batch over all segments and merge into the extended table

Builds prefixed columns (`speaker_id`, `spk_ac_*`, `spk_txt_*`) and composes onto
`all_features_combined_extended.pickle` (created if absent, extended if present). The originals are untouched.

**Note:** this downloads one separated-audio `.flac` per (meeting, speaker) the first time — sizeable but
cached. Run after the single-segment test above looks correct.

### 6e. Tier-1 coupling features (speaker audio ↔ listener motion)

Engagement is the listener *reacting* to the speaker, so these cross-modal features tie the two channels. The
listener (the target person) never speaks in the window, so their response is visual: we read their OpenPose
keypoints (downloaded per meeting, ~25 fps), build a per-frame **upper-body motion-energy** series, and relate it
to the speaker's intensity envelope and pitch-rise events:
- `coup_xcorr_max` / `coup_xcorr_lag` — peak cross-correlation (and its lag) of listener motion vs speaker
  loudness, searching lags where the **listener follows the speaker** (0–1.2 s);
- `coup_corr0` — zero-lag correlation;
- `coup_motion_mean` — listener overall motion in the window;
- `resp_after_rise` — extra listener motion in the 1 s **after** speaker pitch-rise events (responsiveness to
  question/rising intonation — the v2 winner).



In [15]:
import glob

KP_FPS = 25.0   # OpenPose frame rate (verified: frames / flac-duration ≈ 25)

def _keypoint_index_map(meeting, person):
    """frame-index -> json path for a participant's OpenPose folder (cached per person)."""
    cache = _keypoint_index_map.__dict__.setdefault("_c", {})
    key = (meeting, person)
    if key not in cache:
        kp = RAW_DIR / meeting / "openpose" / meeting / f"{meeting}-{person}_keypoints"
        mp = {}
        if kp.is_dir():
            for f in glob.glob(str(kp / "*.json")):
                m = re.search(r"_(\d{12})_keypoints", f)
                if m:
                    mp[int(m.group(1))] = f
        cache[key] = mp
    return cache[key]

def listener_motion_series(meeting, person, start, end):
    """Per-frame upper-body motion energy (mean displacement of confident keypoints) at ~25 fps."""
    download_raw_for_meeting(meeting, what=("keypoints",))   # ~105 MB/meeting, cached
    mp = _keypoint_index_map(meeting, person)
    if not mp:
        return np.array([]), np.array([])
    prev = None; t = []; energy = []
    for idx in range(int(start * KP_FPS), int(end * KP_FPS) + 1):
        f = mp.get(idx)
        if f is None:
            prev = None; continue
        d = json.load(open(f))
        if not d["people"]:
            prev = None; continue
        arr = np.array(d["people"][0]["pose_keypoints_2d"]).reshape(-1, 3)
        if prev is not None:
            conf = (arr[:, 2] > 0.3) & (prev[:, 2] > 0.3)
            if conf.sum() > 0:
                energy.append(float(np.mean(np.linalg.norm(arr[conf, :2] - prev[conf, :2], axis=1))))
                t.append(idx / KP_FPS)
        prev = arr
    return np.array(t), np.array(energy)

def _speaker_rise_times(snd, start, f0min=75, f0max=400, thr=2.0):
    """Times (s) at which a >= thr-semitone cumulative F0 rise completes."""
    pitch = snd.to_pitch(pitch_floor=f0min, pitch_ceiling=f0max)
    f0 = pitch.selected_array["frequency"]; pt = pitch.xs() + start
    v = f0 > 0; vf0 = f0[v]; vt = pt[v]
    if vf0.size < 3:
        return []
    st = 12.0 * np.log2(vf0 / np.median(vf0))
    times = []; cum = 0.0
    for i in range(1, len(st)):
        d = st[i] - st[i - 1]
        cum = (max(cum, 0) + d) if d > 0 else (min(cum, 0) + d)
        if cum >= thr:
            times.append(float(vt[i])); cum = 0.0
    return times

_COUP_KEYS = ["coup_xcorr_max", "coup_xcorr_lag", "coup_corr0", "coup_motion_mean", "resp_after_rise"]

def compute_coupling_features(segment_id, max_lag=1.2):
    """Cross-modal coupling between the speaker's speech and the listener's motion."""
    p = parse_segment_id(segment_id)
    speaker, _ = find_speaker(segment_id)
    if speaker is None:
        return {**{k: np.nan for k in _COUP_KEYS}, "has_coupling": False}
    t, mot = listener_motion_series(p["meeting"], p["person"], p["start"], p["end"])
    if mot.size < 8:
        return {**{k: np.nan for k in _COUP_KEYS}, "has_coupling": False}

    download_raw_for_meeting(p["meeting"], persons=[speaker], what=("audio",))
    flac = raw_audio_path(p["meeting"], speaker)
    sr = sf.info(str(flac)).samplerate
    y, _ = sf.read(str(flac), start=int(p["start"] * sr), stop=int(p["end"] * sr), dtype="float64")
    if y.ndim > 1:
        y = y.mean(axis=1)
    snd = parselmouth.Sound(y, sampling_frequency=sr)
    inten = snd.to_intensity(); iv = inten.values[0]; it = inten.xs() + p["start"]
    iv = np.nan_to_num(iv, nan=float(np.nanmin(iv)))
    env = np.interp(t, it, iv)                         # speaker loudness on listener time grid

    a = (mot - mot.mean()) / (mot.std() + 1e-9)
    b = (env - env.mean()) / (env.std() + 1e-9)
    n = len(a)
    xc = np.correlate(a, b, mode="full") / n
    lags = np.arange(-n + 1, n) / KP_FPS
    win = (lags >= 0) & (lags <= max_lag)              # listener follows speaker
    out = {"has_coupling": True,
           "coup_xcorr_max": float(np.max(xc[win])),
           "coup_xcorr_lag": float(lags[win][np.argmax(xc[win])]),
           "coup_corr0": float(np.corrcoef(a, b)[0, 1]) if n > 1 else 0.0,
           "coup_motion_mean": float(mot.mean()),
           "resp_after_rise": 0.0}

    rises = _speaker_rise_times(snd, p["start"])
    if rises:
        base = mot.mean()
        gains = [mot[(t >= te) & (t <= te + 1.0)].mean() - base
                 for te in rises if mot[(t >= te) & (t <= te + 1.0)].size]
        if gains:
            out["resp_after_rise"] = float(np.mean(gains))
    return out

# quick check on the validated segment
print("coupling demo:", compute_coupling_features("20210323-SP01F_clip_732_737"))

20210323.zip: 100%|██████████| 110M/110M [00:36<00:00, 3.00MB/s] 


[ok] extracted keypoints -> /Users/ducanh/Desktop/CCS2_code/raw_data/20210323/openpose
[skip] 20210323-SP06M.flac already downloaded
coupling demo: {'has_coupling': True, 'coup_xcorr_max': 0.01257531993941259, 'coup_xcorr_lag': 0.28, 'coup_corr0': -0.28406809047721604, 'coup_motion_mean': 5.236138571762276, 'resp_after_rise': -0.8143224458146218}


### 6e. Turn-taking context features (TextGrid)

Cheap interaction-context features from the meeting TextGrid (already cached) — no audio/keypoints needed:
how many people are speaking (liveliness), how long the speaker has been holding the floor leading in
(monologue length), how recently the listener last spoke, simultaneous speech, and position within the meeting
(engagement tends to decay over long meetings).

In [13]:
_CTX_KEYS = ["num_speakers", "speaker_lead_sec", "listener_recent_sec", "meeting_position", "overlap_others"]

def compute_speaker_context_features(segment_id, lead=8.0):
    """Turn-taking / positional context of the segment window from the TextGrid."""
    p = parse_segment_id(segment_id); tg = load_textgrid(p["meeting"])
    s, e = p["start"], p["end"]
    ends = [iv.end_time for t in tg.tiers for iv in getattr(t, "intervals", [])
            if iv.text.strip() and iv.end_time > iv.start_time >= 0]
    meeting_dur = max(ends) if ends else max(e, 1)
    active = [t.name for t in tg.tiers if _tier_speech_intervals(t, s, e)]
    speaker, _ = find_speaker(segment_id)

    def tier_speech(name, a, b):
        t = next((x for x in tg.tiers if x.name == name), None)
        if t is None:
            return 0.0
        return sum(_overlap(a, b, iv.start_time, iv.end_time) for iv in _tier_speech_intervals(t, a, b))

    return {
        "has_context": True,
        "num_speakers": len(active),
        "speaker_lead_sec": tier_speech(speaker, s - lead, e) if speaker else 0.0,
        "listener_recent_sec": tier_speech(p["person"], s - lead, s),
        "meeting_position": float(s / max(meeting_dur, 1)),
        "overlap_others": sum(tier_speech(n, s, e) for n in active if n != speaker),
    }

print("context demo:", compute_speaker_context_features("20210323-SP01F_clip_732_737"))

context demo: {'has_context': True, 'num_speakers': 1, 'speaker_lead_sec': 7.657999999999447, 'listener_recent_sec': 0, 'meeting_position': 0.5623510576348374, 'overlap_others': 0}


In [16]:
from tqdm.auto import tqdm

def speaker_feature_row(segment_id):
	ac = compute_speaker_acoustic_features(segment_id)
	tx = compute_speaker_text_features(segment_id)
	spk, ov = find_speaker(segment_id)
	row = {"speaker_id": spk, "speaker_overlap_sec": ov, "spk_has_speaker": ac["has_speaker"]}
	row.update({f"spk_ac_{k}": v for k, v in ac.items() if k != "has_speaker"})
	coup = compute_coupling_features(segment_id)
	row.update({f"spk_txt_{k}": v for k, v in tx.items() if k != "has_speaker"})
	row.update({f"spk_coup_{k}": v for k, v in coup.items() if k != "has_coupling"})
	row["spk_has_coupling"] = coup["has_coupling"]
	return row

# Compose onto the extended table if it exists, else start from the base table.
ext_path = REPO_ROOT / "all_features_combined_extended.pickle"
src_path = ext_path if ext_path.exists() else (REPO_ROOT / "all_features_combined.pickle")
with open(src_path, "rb") as f:
	df = pickle.load(f)

tqdm.pandas(desc="speaker features")
spk_df = pd.DataFrame(list(df["segment_id"].progress_apply(speaker_feature_row)))
spk_df.index = df.index
out_df = pd.concat([df.drop(columns=spk_df.columns, errors="ignore"), spk_df], axis=1)

with open(ext_path, "wb") as f:
	pickle.dump(out_df, f)
out_df.to_csv(REPO_ROOT / "all_features_combined_extended.csv", index=False)
print(f"saved {ext_path.name} with {spk_df.shape[1]} new speaker columns")
print("segments with an identified speaker:", int(out_df["spk_has_speaker"].sum()), "/", len(out_df))
out_df[["segment_id", "speaker_id", "spk_ac_f0_mean", "spk_txt_word_count", "spk_txt_sent_compound"]].head()

speaker features:   0%|          | 2/549 [00:00<00:47, 11.44it/s]

[skip] 20210323-SP01F.flac already downloaded
[skip] 20210323.zip already downloaded
[skip] 20210323-SP01F.flac already downloaded
[skip] 20210323-SP01F.flac already downloaded


speaker features:   1%|          | 4/549 [00:00<00:43, 12.39it/s]

[skip] 20210323.zip already downloaded
[skip] 20210323-SP01F.flac already downloaded
[skip] 20210323-SP01F.flac already downloaded
[skip] 20210323.zip already downloaded
[skip] 20210323-SP01F.flac already downloaded
[skip] 20210323-SP01F.flac already downloaded


speaker features:   1%|          | 6/549 [00:00<00:50, 10.70it/s]

...

[skip] 20220916-SP04F.flac already downloaded
[skip] 20220916-SP01F.flac already downloaded
[skip] 20220916.zip already downloaded
[skip] 20220916-SP01F.flac already downloaded
[skip] 20220916-SP01F.flac already downloaded
[skip] 20220916.zip already downloaded
[skip] 20220916-SP01F.flac already downloaded


speaker features: 100%|██████████| 549/549 [32:22<00:00,  3.54s/it]

[skip] 20220916-SP01F.flac already downloaded
[skip] 20220916.zip already downloaded
[skip] 20220916-SP01F.flac already downloaded
[skip] 20220916-SP01F.flac already downloaded
[skip] 20220916.zip already downloaded
[skip] 20220916-SP01F.flac already downloaded


saved all_features_combined_extended.pickle with 39 new speaker columns
segments with an identified speaker: 531 / 549


,segment_id,speaker_id,spk_ac_f0_mean,spk_txt_word_count,spk_txt_sent_compound
0,20210323-SP07F_clip_378_383,SP01F,188.629870,21.0,-0.0762
1,20210323-SP07F_clip_1113_1118,None,NaN,0.0,0.0000
2,20210323-SP07F_clip_1074_1079,SP01F,182.983159,13.0,0.5232
3,20210323-SP07F_clip_1182_1187,SP01F,180.814147,10.0,-0.0762
4,20210323-SP07F_clip_255_260,SP01F,158.339017,9.0,0.0000


## 9. Use the new features downstream

`clustering.ipynb` / `clustering_alignment.ipynb` load `all_features_combined.pickle`. To cluster with the new feature(s), point them at `all_features_combined_extended.pickle` instead (and add `FEATURE_NAME` to the
feature columns they stack). Labels still come from `label_df_3` (`majority_label_3`), joined on `segment_id`.